<img src="https://raw.githubusercontent.com/mctools/ncrystal-logo/main/png/logo-compact/ncrystal-logo-compact-64h.png" alt="NCrystal" width="158">

# Using NCrystal as a backend in full-fledged Monte Carlo simulation applications

In [ ]:
#Install software on Google Colab:
%pip -q install ncrystal
#Import NCrystal:
import NCrystal as NC
assert NC.version_num >= 4004006, "NCrystal version 4.4.6 or later is needed"
NC.test() #< quick test that the installation works

<a id="mcintro-intro"></a>

## Introduction
In this notebook we discuss how the full potential of NCrystal is realised when used as a physics engine for applications which track neutrons through complex geometries, where they can undergo multiple scattering. In particular we will mention how NCrystal is used in applications like McStas, OpenMC, or Geant4.

The other notebooks of this section show this in practice, starting with NCrystal's own simple [MiniMC](https://tkittel.github.io/ncrystal-notebooks/notebooks/minimc.html) engine. The examples for OpenMC and McStas are linked in their sections below.

<a id="mcintro-othercodes"></a>

## Using NCrystal with other codes
Apart from being used by itself, the original reason for developing NCrystal was indeed to serve as a backend engine for existing Monte Carlo codes, providing a consistent high-fidelity neutron scattering physics to those. One advantage here is of course that it allows both developers and users across all of the supported codes to collaborate on the neutron scattering models. In this notebook, we will focus on three such applications that are all well-supported, widely used, and open-source: OpenMC, McStas, and Geant4.

<a id="mcintro-openmc"></a>

### OpenMC
Using NCrystal materials in OpenMC is supported since OpenMC release 13.3, and since OpenMC release 15.2 from spring 2025 it does not even require a custom build of OpenMC. The integration relies on a nice simple syntax in the Python API (see also https://docs.openmc.org/en/stable/usersguide/materials.html#adding-ncrystal-materials):

```python
mat = openmc.Material.from_ncrystal('Polyethylene_CH2.ncmat;temp=50C')
```

which when used in a complete OpenMC project, results in the following material entry being added to the `materials.xml` produced:

```xml
  <material cfg="Polyethylene_CH2.ncmat;temp=50C" id="1" temperature="323.15">
    <density units="g/cm3" value="0.92" />
    <nuclide ao="0.66656284" name="H1" />
    <nuclide ao="0.00010382666666666666" name="H2" />
    <nuclide ao="0.32964066666666664" name="C12" />
    <nuclide ao="0.003692666666666666" name="C13" />
  </material>
```

Temperature, density and material composition were all created automatically from the cfg-string, and the cfg-string itself was also encoded. Upon launching the simulation with the OpenMC binary executable `openmc`, it will handle the material as usual, except that low-energy neutron scattering physics (currently defined as $E<5$ eV) will be provided by the algorithms in NCrystal.

A few issues might warrant attention:

1. If you try to assemble the above xml manually, it is rather unlikely that you will get the base densities and compositions right. It is safest to stick to letting the Python API compose the xml for you.
2. After creation with `mat=openmc.Material.from_ncrystal(..)`, you can not use the usual OpenMC API to modify the material density, temperature, or composition. So be sure to reflect the final desired material inside the NCrystal cfg-string.

For reference, here is a complete small example of an OpenMC project in Python using NCrystal.

```python
import openmc
# Materials
openmc_mat = openmc.Material.from_ncrystal('Polyethylene_CH2.ncmat;temp=50C')
# Geometry
s1 = openmc.Sphere(r=10, boundary_type='vacuum')
c1 = openmc.Cell(region=-s1, fill=openmc_mat)
geometry = openmc.Geometry([c1])
# Execution settings
settings = openmc.Settings()
settings.source = openmc.Source(energy=openmc.stats.Discrete(x=[10.0], p=[1.0]))
settings.run_mode = 'fixed source'
settings.batches = 10
settings.particles = 10000
# Write xml files
model = openmc.model.Model(geometry=geometry, settings=settings)
model.export_to_xml()
#Check resulting materials.xml (we could of course actually RUN OpenMC here instead)
import pathlib
print(pathlib.Path('materials.xml').read_text())
```

Credits: The OpenMC-NCrystal bindings originated with the NCrystal developers (esp. J.I. Marquez Damian), but were facilitated along the way by useful technical assistance and a welcoming attitude from the OpenMC developers (esp. Paul Romano).

For a complete example which runs OpenMC with NCrystal materials, see [NCrystal materials in OpenMC](https://tkittel.github.io/ncrystal-notebooks/notebooks/openmc.html).

<a id="mcintro-mcstas"></a>

### McStas

There are three ways to run McStas simulations:

* **McStas** itself, with its `mcrun` command (which translates an instrument file, written in the McStas instrument language, into C code with the `mcstas` command, compiles it, and runs it).
* **[McStasScript](https://github.com/PaNOSC-ViNYL/McStasScript)**, a Python package for building instruments in Python, which then runs them with `mcrun` (so it needs a McStas installation).
* **[mccode-antlr](https://github.com/mccode-dev/mccode-antlr)**, a Python package which implements the McStas instrument language itself, and only uses the components and data files of McStas, which it downloads when needed. It can be installed with pip and needs no McStas installation (but a C compiler). Instruments can be given as instrument code, or built step by step with Python objects.

The other notebooks of this section show examples [with McStas](https://tkittel.github.io/ncrystal-notebooks/notebooks/mcstasexample.html), and with mccode-antlr [from instrument code](https://tkittel.github.io/ncrystal-notebooks/notebooks/mcstasantlr.html) and [with an instrument built in Python](https://tkittel.github.io/ncrystal-notebooks/notebooks/mcstasantlrapi.html).

All three use the same McStas components, and several of these use NCrystal materials, given as NCrystal cfg-strings:

* `NCrystal_sample`: a sample, filter, monochromator, window, etc. in a simple shape (box, cylinder or sphere), with the full NCrystal physics (see below).
* The Union components, for complex geometries, with multiple scattering between materials: `NCrystal_process` provides an NCrystal scattering process, to be combined with other processes in a Union material (`Union_make_material`), and since McStas 3.9.0, `Union_NCrystal_material` makes an entire Union material directly from a cfg-string.
* `NCrystal_filter` (since McStas 3.9.0): a filter or window which only attenuates the beam, according to the total cross section of the material (scattering plus absorption). It is simple, fast, and can also run on GPUs.
* `PowderN` and `Single_crystal` (since McStas 3.9.0) accept an NCrystal cfg-string as their `reflections` parameter, and then take the reflections, unit cell and cross sections from NCrystal. They only model Bragg diffraction, absorption and incoherent scattering.

For instance, a Union material is defined from a cfg-string with:
```c
COMPONENT myAl = Union_NCrystal_material(cfg="stdlib::Al_sg225.ncmat;temp=250K")
AT (0,0,0) ABSOLUTE
```
and Union volumes then use it by its name, with `material_string="myAl"`.

The `NCrystal_sample` component embeds NCrystal material simulations in simple shapes. For instance, a cylinder with a radius of 1 cm and a height of 5 cm of powdered sapphire is:
```c
COMPONENT mysample = NCrystal_sample(cfg="stdlib::Al2O3_sg167_Corundum.ncmat",
                                     radius=0.01, yheight=0.05)
AT (0, 0, 0) RELATIVE PREVIOUS
```
For more documentation about the NCrystal_sample component, run:
```bash
$> mcdoc NCrystal_sample
```
Or consult the documentation online at https://www.mcstas.org/download/components/

*Credits: The integration into McStas, Union, and McStasScript is largely a result of efforts by Peter Willendrup and Mads Bertelsen working in close collaboration with the NCrystal developers.*

<a id="mcintro-geant4"></a>

### Geant4

Geant4-bindings allow NCrystal cfg-strings to be used directly in Geant4 material creation code, through the usage of specific helper functions:

```c++
//Include the relevant header:
#include "G4NCrystal/G4NCrystal.hh"

//...

//Create materials directly from cfg-strings:
G4Material * mat_aluminium = G4NCrystal::createMaterial("Al_sg225.ncmat");

//...

//Currently the NCrystal process must then be injected in the following way:
g4runManager->Initialize();
G4NCrystal::installOnDemand();
g4runManager->BeamOn(1000);
```
The last part about injecting a specific NCrystal process is a bit of a technical workaround, and the NCrystal developers are in contact with the Geant4 Hadronic working group (in particular thanks to Alberto Ribon) about a better solution. But it works for now in non-multithreaded Geant4.

For a fully fledged example, see:

https://github.com/mctools/ncrystal/blob/master/examples/ncrystal_example_g4sim.cc

<a id="mcintro-others"></a>

### Others

While OpenMC, Geant4, and McStas are currently the best known and supported frameworks, other emerging applications are currently integrated with NCrystal, including the Cinema framework from CSNS (Xiao Xiao Cai, et al.), [ANTS2](https://github.com/andrmor/ANTS2) from LIP-Coimbra (Andrei Morozov, et al.), and [TOUCANS](https://doi.org/10.1016/j.nima.2023.168190) from CEA (Loïc Thulliez, et al.).

It is also possible to benefit partly from NCrystal by using several of the conversion mechanisms we have for creating data-files for other codes. In all cases though, a lot of the physics is lost in the conversion so if at all possible, you should try to use NCrystal directly:
* You can use the `ncrystal_ncmat2hkl` commandline tool to create `.lau` and `.laz` files for various non-NCrystal Bragg diffraction components in McStas. The same feature can also be accessed in the NCrystal Python API by using the `cfgstr_2_hkl` function from the `NCrystal.mcstasutils` module.
* You can use the `ncrystal_ncmat2endf` command line tool (or the `NCrystal.ncmat2endf` module in Python) to create ENDF files from NCrystal materials, which NJOY can convert to ACE files for codes like OpenMC or MCNP. See the [nuclear data export](https://tkittel.github.io/ncrystal-notebooks/notebooks/endfexport.html) notebooks.
* You can use the NJOY-NCrystal project to create ENDF files from NCrystal data files, which can then be used in traditional codes like MCNP, etc. For more information see:
  * https://github.com/highness-eu/NJOY-NCrystal-Library
  * K. Ramic, et al. "NJOY+NCrystal: an open-source tool for creating thermal neutron scattering libraries"  Nuclear Instruments and Methods in Physics Research Section A: Accelerators, Spectrometers, Detectors and Associated Equipment 1027 (2022) 166227, https://doi.org/10.1016/j.nima.2021.166227
